In [3]:
import os
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# CONFIGURATION
# ============================================================
BASE_DIR = "../data/cve/global_context_daily"
OUTPUT_GLOBAL_MONTH = os.path.join(BASE_DIR, "global_monthly_summary.csv")
OUTPUT_GLOBAL_DAY = os.path.join(BASE_DIR, "global_daily_summary.csv")

OUTPUT_FIG_PDF = os.path.join(BASE_DIR, "cve_matching.pdf")
OUTPUT_FIG_PNG = os.path.join(BASE_DIR, "cve_matching.png")

ANNOTATION_THRESHOLD = 5


# ============================================================
# 1. MERGE DAILY AND MONTHLY SUMMARIES
# ============================================================
def merge_context_summaries(base_dir: str):
    daily_files = sorted(glob.glob(os.path.join(base_dir, "*_daily_summary.csv")))

    print("daily summary files:", len(daily_files))

    if len(daily_files) == 0:
        raise FileNotFoundError(f"No daily summary files found in {base_dir}")

    daily_list = []
    for f in daily_files:
        df = pd.read_csv(f)
        daily_list.append(df)

    global_daily = pd.concat(daily_list, ignore_index=True)

    # Ensure expected columns exist
    required_cols = {"day", "month", "cve_id", "count"}
    missing = required_cols - set(global_daily.columns)
    if missing:
        raise ValueError(f"Missing required columns in daily summaries: {missing}")

    global_daily = global_daily.sort_values(["month", "day", "count"], ascending=[True, True, False])

    global_month = (
        global_daily.groupby(["month", "cve_id"], as_index=False)["count"]
        .sum()
        .sort_values(["month", "count"], ascending=[True, False])
    )

    global_daily.to_csv(OUTPUT_GLOBAL_DAY, index=False)
    global_month.to_csv(OUTPUT_GLOBAL_MONTH, index=False)

    print("global summaries completed")
    print("written to:")
    print(OUTPUT_GLOBAL_DAY)
    print(OUTPUT_GLOBAL_MONTH)

    return global_daily, global_month


# ============================================================
# 2. DATA CLEANING AND PIVOT
# ============================================================
def split_cve(cve_str):
    if pd.isna(cve_str):
        return [cve_str]

    cve_str = str(cve_str).strip().strip('"').strip("'")

    # Split possible multi-CVE fields
    if re.search(r"[,|]", cve_str):
        parts = re.split(r"[,|]\s*", cve_str)
        return [
            p.strip().strip('"').strip("'").replace("\u202f", "").replace("\ufeff", "")
            for p in parts
            if p and p.strip()
        ]

    return [cve_str.replace("\u202f", "").replace("\ufeff", "")]


def generate_heatmap_data(csv_path: str):
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError as e:
        print(f"FATAL ERROR: CSV file not found at {csv_path}")
        raise e

    if not {"month", "cve_id", "count"}.issubset(df.columns):
        raise ValueError("Input CSV must contain: month, cve_id, count")

    df["cve_id"] = df["cve_id"].apply(split_cve)
    df_exploded = df.explode("cve_id").reset_index(drop=True)

    df_exploded["cve_id"] = df_exploded["cve_id"].astype(str).str.strip()
    df_exploded = df_exploded[df_exploded["cve_id"] != ""].copy()

    df_pivot = (
        df_exploded.groupby(["cve_id", "month"], as_index=False)["count"]
        .sum()
    )

    month_order = sorted(df_pivot["month"].unique())

    pivot_table = df_pivot.pivot_table(
        index="cve_id",
        columns="month",
        values="count",
        fill_value=0,
        aggfunc="sum"
    )
    pivot_table = pivot_table.reindex(columns=month_order)
    pivot_table.index.name = None

    def get_sort_key(row):
        positive_months = row[row > 0]
        first_month = positive_months.index[0] if len(positive_months) > 0 else "9999-99"
        total_count = row.sum()
        return (first_month, -total_count)

    sort_keys = pivot_table.apply(get_sort_key, axis=1)
    pivot_table = pivot_table.loc[sort_keys.sort_values().index]

    # log10(count), map zero to log10(1)=0
    pivot_log = np.log10(pivot_table.replace(0, 1))

    return pivot_table, pivot_log


# ============================================================
# 3. PLOT
# ============================================================
def annotate_cell(raw_val):
    if raw_val >= 1000:
        return f"{raw_val/1000:.0f}K"
    if raw_val >= ANNOTATION_THRESHOLD:
        return f"{raw_val:.0f}"
    return ""


def plot_cve_heatmap(pivot_table: pd.DataFrame, pivot_log: pd.DataFrame):
    num_cves = len(pivot_table)
    num_months = len(pivot_table.columns)

    fig_height = max(8, num_cves * 0.38)
    fig_width = max(8, num_months * 1.2 + 3)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    im = ax.imshow(
        pivot_log.values,
        aspect="auto",
        interpolation="nearest",
        origin="upper",
        cmap="Blues",
        vmin=0,
        vmax=pivot_log.values.max(),
    )

    annot_labels = np.asarray([
        [annotate_cell(v) for v in row]
        for row in pivot_table.values
    ])

    for i in range(pivot_table.shape[0]):
        for j in range(pivot_table.shape[1]):
            label = annot_labels[i, j]
            if label:
                ax.text(
                    j,
                    i,
                    label,
                    ha="center",
                    va="center",
                    fontsize=14,
                    color="dimgray",
                )

    month_labels = pivot_table.columns.tolist()
    filtered_labels = [m if i % 2 == 1 else "" for i, m in enumerate(month_labels)]

    ax.set_xticks(np.arange(len(month_labels)))
    ax.set_xticklabels(filtered_labels, rotation=0, fontsize=18)

    ax.set_yticks(np.arange(len(pivot_table.index)))
    ax.set_yticklabels(pivot_table.index.tolist(), fontsize=18)

    ax.set_xlabel("Month", fontsize=22)
    ax.tick_params(axis="x", pad=10)

    cbar = fig.colorbar(im, ax=ax)
    cbar.ax.tick_params(labelsize=16)
    cbar.set_label("Total Attack Count (Log10 Scale)", fontsize=20)

    plt.tight_layout()
    plt.savefig(OUTPUT_FIG_PDF, dpi=300, format="pdf")
    plt.savefig(OUTPUT_FIG_PNG, dpi=300, format="png")
    plt.close()

    print(f"\nSaved heatmap to:")
    print(OUTPUT_FIG_PDF)
    print(OUTPUT_FIG_PNG)


# ============================================================
# 4. EXECUTION
# ============================================================
if __name__ == "__main__":
    merge_context_summaries(BASE_DIR)
    pivot_table_raw, pivot_table_log = generate_heatmap_data(OUTPUT_GLOBAL_MONTH)
    plot_cve_heatmap(pivot_table_raw, pivot_table_log)

daily summary files: 332
global summaries completed
written to:
../data/cve/global_context_daily/global_daily_summary.csv
../data/cve/global_context_daily/global_monthly_summary.csv

Saved heatmap to:
../data/cve/global_context_daily/cve_matching.pdf
../data/cve/global_context_daily/cve_matching.png
